## Lab 11  
**Scaling Machine Learning with PySpark: Spark Execution, Distributed DataFrames, and MLlib Pipelines on Dataproc**


## Learning Objectives

By the end of this lab, you will be able to:

1. **Explain** why large-scale machine learning is fundamentally a systems problem involving memory, I/O, shuffles, iteration cost, and fault tolerance.
2. **Implement** PySpark workflows that distinguish lazy transformations from actions and **interpret** how DAG stages and shuffles affect runtime.
3. **Validate** a real dataset in Spark using schema checks, label audits, missing-value checks, duplicate checks, and partition diagnostics.
4. **Build** leakage-safe Spark MLlib pipelines for multiclass classification using `DataFrame`-based feature engineering and model training.
5. **Evaluate** and **compare** a multinomial logistic-regression baseline and a tuned random-forest model using multiclass metrics.
6. **Profile** caching, repartitioning, and train-validation tuning choices and **relate** them to memory, throughput, and wall-clock cost.
7. **Interpret** confusion patterns and feature-importance diagnostics and **propose** next modeling steps.
8. **Persist**, **reload**, and **monitor** a Spark pipeline in a way that supports reproducibility and production reliability.


## Tasks

- Configure a reproducible PySpark environment on Dataproc and record experiment metadata.
- Acquire the UCI Covertype dataset programmatically with a fallback download path.
- Convert raw data to a Spark `DataFrame`, optionally stage it as Parquet, and inspect partitions.
- Validate schema, labels, missingness, duplicates, and class balance.
- Use small RDD and `DataFrame` examples to inspect lazy evaluation, actions, and wide versus narrow transformations.
- Build a leakage-safe multinomial logistic-regression baseline in an MLlib `Pipeline`.
- Tune and compare a random-forest pipeline with `TrainValidationSplit`.
- Diagnose results with a confusion matrix, dominant error pairs, and feature importances.
- Save and reload the best pipeline model and propose a practical monitoring plan.


**Note:** In this programming lab, you are required to execute all cells step by step and focus on understanding the methodology and the outcomes at each step. Answering the questions included in this lab (pre-coding and post-execution interpretation questions) is optional and is not required for the completion of the programming lab.

## References

### Official Documentation
 
- [`pyspark.sql.SparkSession`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html)
- [`pyspark.sql.DataFrame`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)
- [`pyspark.ml.Pipeline`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.Pipeline.html)
- [`pyspark.ml.feature.StringIndexer`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html)
- [`pyspark.ml.feature.VectorAssembler`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html)
- [`pyspark.ml.feature.StandardScaler`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StandardScaler.html)
- [`pyspark.ml.classification.LogisticRegression`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegression.html)
- [`pyspark.ml.classification.RandomForestClassifier`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.RandomForestClassifier.html)
- [`pyspark.ml.evaluation.MulticlassClassificationEvaluator`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.MulticlassClassificationEvaluator.html)
- [`pyspark.ml.tuning.ParamGridBuilder`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.ParamGridBuilder.html)
- [`pyspark.ml.tuning.TrainValidationSplit`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.TrainValidationSplit.html)
- [`sklearn.datasets.fetch_covtype`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html)

### References

- [Spark ML classification and regression guide](https://spark.apache.org/docs/latest/ml-classification-regression.html)
- [Spark model-selection and tuning guide](https://spark.apache.org/docs/latest/ml-tuning.html)

### Python Commands and Quick Reference

- `spark.createDataFrame(pdf)` — move a pandas `DataFrame` into Spark
- `df.repartition(n)` — change partition count
- `df.persist(StorageLevel.MEMORY_AND_DISK)` — cache a `DataFrame` with spill support
- `df.randomSplit([0.8, 0.2], seed=42)` — make reproducible train/test splits
- `Pipeline(stages=[...])` — chain preprocessing and modeling steps
- `model.transform(df)` — score a `DataFrame`
- `evaluator.evaluate(pred_df, {evaluator.metricName: "accuracy"})` — compute alternate metrics
- `ParamGridBuilder().addGrid(...).build()` — define a tuning search space
- `TrainValidationSplit(...).fit(train_df)` — run faster hyperparameter selection
- `PipelineModel.load(path)` — reload a persisted fitted pipeline


## Workflow Overview

1. Configure imports, seeds, Spark session settings, and a reproducibility manifest.
2. Acquire Covertype programmatically, create a Spark `DataFrame`, and stage it for scalable reuse.
3. Validate schema and labels, inspect class balance, and diagnose partition distribution and caching behavior.
4. Use small RDD and `DataFrame` examples to inspect lazy execution, actions, and wide versus narrow transformations.
5. Build a leakage-safe baseline MLlib pipeline with `StringIndexer`, `VectorAssembler`, `StandardScaler`, and multinomial `LogisticRegression`.
6. Improve the model with `RandomForestClassifier`, `ParamGridBuilder`, and `TrainValidationSplit`, then compare predictive gains with systems cost.
7. Diagnose model behavior using a confusion matrix, dominant error pairs, and feature importances.
8. Persist the best fitted pipeline, reload it for inference, and design a monitoring checklist for production use.


## Dataset

**Primary dataset:** Covertype (UCI / scikit-learn fetcher)

- **Dataset name:** Covertype
- **Load link:** `sklearn.datasets.fetch_covtype(as_frame=True)`
- **Direct download link:** `https://archive.ics.uci.edu/static/public/31/covertype.zip`
- **Dataset page:** https://archive.ics.uci.edu/dataset/31/covertype
- **Description:** Predict forest cover type from 54 cartographic attributes such as elevation, slope, hillshade, wilderness area indicators, and soil-type indicators.
- **Features:** 54 integer/binary predictors
- **Target:** `Cover_Type` with 7 classes
- **Size:** 581,012 rows, 54 features
- **Licensing / usage note:** UCI lists the dataset under CC BY 4.0..

**Fallback acquisition path:** direct UCI ZIP download followed by pandas loading and Spark conversion.

This dataset is large enough to motivate partitioning, caching, and distributed tuning, yet still practical for an instructional Dataproc lab.


### Step 1 — Setup & Reproducibility

### Pre-coding questions

1. Which notebook outputs are most likely to change when the random seed changes, and which outputs should remain identical?    
Train and test splits, model predictions, and evaluation metrics may change when the random seed changes. Dataset size, schema, feature names, and basic data validation results should be the same because they do not depend on random sampling.

2. Why can two Spark runs still differ slightly across machines even when `random_state` and Spark seeds are fixed?     
Different machines can have different Spark and Python versions, hardware, cluster configurations, and partition layouts. Distributed processing can also execute tasks in a different order, which may cause small differences in results and execution times.

3. What experiment metadata should be logged before any model fitting occurs?       
I would record the random seed, Spark version, Python version, cluster master, default parallelism, shuffle partition count, dataset location, and model storage location. The model parameters and train/test split settings should also be recorded so the experiment can be reproduced.



### Function/class selection and parameter exploration

Use:

- `pyspark.sql.SparkSession`
- `numpy.random.seed`
- `pyspark.sql.DataFrame.randomSplit`
- `pyspark.ml.Pipeline`

Explore with `help()` or notebook `?`:

- `SparkSession.builder.appName(...).getOrCreate()`
- `DataFrame.randomSplit(weights, seed)`
- `Pipeline(stages=...)`

Explain the effect of:

- `seed` / `random_state` on split reproducibility
- `spark.sql.shuffle.partitions` on parallelism and shuffle overhead
- `appName` on traceability in Spark UI and logs
- `stages` ordering inside `Pipeline` on leakage control and correctness



> **Think like an ML Systems engineer:** 

Reproducibility is not only about the model seed. It is also about cluster settings, data location, library versions, and serialization boundaries.


In [1]:

# Step 1: Setup & reproducibility
import os
import sys
import time
import json
import zipfile
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_covtype

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

spark = (
    SparkSession.builder
    .appName("Lab11-ScalingML-PySpark")
    .config("spark.driver.memory", "6g")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
sc = spark.sparkContext

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", max(64, sc.defaultParallelism * 2))

STAGE_URI = os.environ.get("LAB11_STAGE_URI", "file:///tmp/lab11_covtype_parquet")
MODEL_URI = os.environ.get("LAB11_MODEL_URI", "file:///tmp/lab11_best_pipeline_model")

print("Spark version:", spark.version)
print("Master:", sc.master)
print("Application ID:", sc.applicationId)
print("Default parallelism:", sc.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))


ModuleNotFoundError: No module named 'pyspark'

In [ ]:
run_manifest = {
    "random_state": RANDOM_STATE,
    "spark_version": spark.version,
    "python_version": sys.version.split()[0],
    "master": sc.master,
    "default_parallelism": sc.defaultParallelism,
    "shuffle_partitions": int(spark.conf.get("spark.sql.shuffle.partitions")),
    "stage_uri": STAGE_URI,
    "model_uri": MODEL_URI,
}
pd.Series(run_manifest)

### Post-execution interpretation questions

1. Which printed setup values would you check first if two students report different train/test metrics?        
I would first check the random seed, Spark version, Python version, master, default parallelism, and shuffle partition count. I would also check whether both students used the same dataset and train/test split settings.

2. What is a limitation of using only seeds and versions for reproducibility? What should be added next?        
Seeds and versions do not fully describe the environment or the data used in an experiment. I would also record the dataset source and version, cluster configuration, storage locations, preprocessing settings, and model parameters.

3. From a systems perspective, what is the risk of setting `spark.sql.shuffle.partitions` much too high or much too low?        
If it is too high, Spark may create many small tasks and spend extra time managing them. If it is too low, partitions may become too large, reducing parallelism and potentially causing memory problems or slow execution.


### Step 2 — Data Acquisition & Staging

### Pre-coding questions

1. Why is programmatic dataset acquisition preferable to distributing a hidden local CSV with the notebook?         
Programmatic acquisition makes the dataset source clear and allows other users to retrieve the data themselves. It also makes the notebook easier to reproduce without relying on a file that may only exist on one computer.

2. Why is Parquet usually a better storage format than raw CSV for iterative Spark workflows?           
Parquet is a columnar format that allows Spark to read only the columns needed for a task. It also stores schema information and is generally more efficient for repeated analytical queries than CSV.

3. What tradeoff appears when you convert a pandas `DataFrame` to Spark on the driver first and only later stage it for distributed reuse?      
The entire dataset must first fit in the driver's memory as a pandas DataFrame. This makes the initial loading process less scalable, even though converting it to Spark and staging it as Parquet makes later processing more efficient.



### Function/class selection and parameter exploration

Use:

- `sklearn.datasets.fetch_covtype`
- `SparkSession.createDataFrame`
- `pyspark.sql.DataFrame.repartition`
- `pyspark.sql.DataFrameWriter.parquet`

Explore with `help()` or notebook `?`:

- `fetch_covtype(as_frame=True)`
- `spark.createDataFrame(...)`
- `DataFrame.repartition(numPartitions)`
- `DataFrame.write.mode("overwrite").parquet(path)`

Explain the effect of:

- `as_frame` on schema convenience and driver-memory use
- `numPartitions` on task parallelism and partition size
- `mode="overwrite"` on reproducibility versus safety
- storage path choice (`file:///...` versus `gs://...`) on portability and distributed access


In [ ]:

# Step 2: Column definitions for direct-download fallback
FEATURE_COLUMNS = [
    "Elevation", "Aspect", "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
] + [f"Wilderness_Area_{i}" for i in range(4)] + [f"Soil_Type_{i}" for i in range(40)]

ALL_COLUMNS = FEATURE_COLUMNS + ["Cover_Type"]

DATA_DIR = Path("/tmp/lab11_covtype")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Feature count:", len(FEATURE_COLUMNS))
print("Data directory:", DATA_DIR)


In [ ]:
def load_covtype_pandas():
    try:
        bunch = fetch_covtype(as_frame=True)
        pdf = bunch.frame.copy()
        pdf.columns = [str(c).replace(" ", "_") for c in pdf.columns]
        if "Cover_Type" not in pdf.columns and "target" in pdf.columns:
            pdf = pdf.rename(columns={"target": "Cover_Type"})
        return pdf, "Loaded with sklearn.datasets.fetch_covtype(as_frame=True)"
    except Exception as exc:
        zip_url = "https://archive.ics.uci.edu/static/public/31/covertype.zip"
        zip_path = DATA_DIR / "covertype.zip"
        urlretrieve(zip_url, zip_path)

        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)

        gz_path = DATA_DIR / "covtype.data.gz"
        pdf = pd.read_csv(gz_path, header=None, names=ALL_COLUMNS)
        return pdf, f"Loaded from UCI ZIP fallback after sklearn fetch failed: {type(exc).__name__}"

In [ ]:

# Step 2: Acquire the dataset, convert to Spark, and stage for scalable reuse
pdf, data_source_note = load_covtype_pandas()

print(data_source_note)
print("Pandas shape:", pdf.shape)
display(pdf.head())

spark_raw = spark.createDataFrame(pdf)
TARGET_PARTITIONS = max(sc.defaultParallelism * 2, 8)

try:
    (spark_raw
     .repartition(TARGET_PARTITIONS)
     .write
     .mode("overwrite")
     .parquet(STAGE_URI))
    df = spark.read.parquet(STAGE_URI)
    acquisition_note = f"Using staged Parquet at {STAGE_URI}"
except Exception as exc:
    print(f"Parquet staging skipped: {type(exc).__name__}: {exc}")
    df = spark_raw.repartition(TARGET_PARTITIONS).persist(StorageLevel.MEMORY_AND_DISK)
    _ = df.count()
    acquisition_note = "Using persisted in-memory Spark DataFrame"

dataset_metadata = {
    "dataset_name": "Covertype",
    "rows": int(df.count()),
    "n_features": len(FEATURE_COLUMNS),
    "target_name": "Cover_Type",
    "source_note": data_source_note,
    "spark_note": acquisition_note,
}
pd.Series(dataset_metadata)


### Post-execution interpretation questions

1. Which part of this acquisition path is least scalable, and why is it still acceptable in this lab?           
Loading the entire dataset into pandas before converting it to Spark is the least scalable part because the data must fit in the driver's memory. It is acceptable here because the Covertype dataset is small enough for the instructional environment and provides a straightforward way to introduce Spark.

2. Why is it useful to stage data once and then reuse the staged representation in later steps?         
Staging the data as Parquet avoids repeatedly downloading and converting the original dataset. It also allows later Spark operations to read a consistent representation of the data more efficiently.

3. From a systems perspective, when should `file:///tmp/...` be replaced by a cloud path such as `gs://...`?        
A cloud storage path should be used when the data needs to be shared across multiple cluster workers or reused across different Dataproc sessions. Local temporary storage is less portable and may disappear when the environment is restarted.


### Step 3 — Data Validation & Partition Diagnostics

### Pre-coding questions

1. What failure mode can occur if a label column silently changes type or range before model training?          
The model could interpret the labels incorrectly, fail during training, or produce invalid evaluation results. Checking the label type and expected values helps ensure that the classification problem is being handled correctly.

2. Why should partition diagnostics be part of data validation in Spark, not only schema validation?        
A correct schema does not guarantee that the data is distributed evenly across partitions. Uneven partitions can create stragglers, where some tasks take much longer than others and slow down the entire job.

3. What would happen if you cached a very large `DataFrame` without enough executor memory?     
Spark could run out of available memory and need to spill cached data to disk. This can reduce performance and potentially cause slowdowns or failures if storage resources are also insufficient.



### Function/class selection and parameter exploration

Use:

- `DataFrame.printSchema`
- `DataFrame.groupBy`
- `DataFrame.dropDuplicates`
- `DataFrame.repartition`
- `DataFrame.persist`
- `StorageLevel.MEMORY_AND_DISK`

Explore with `help()` or notebook `?`:

- `DataFrame.dropDuplicates(subset=None)`
- `DataFrame.persist(storageLevel)`
- `StorageLevel.MEMORY_AND_DISK`

Explain the effect of:

- `subset` in `dropDuplicates` on duplicate semantics
- `storageLevel` on memory use and spill behavior
- `repartition` on shuffle cost and future parallelism
- label-domain checks on evaluation validity


In [ ]:

# Step 3: Basic schema inspection
df.printSchema()
print("Current partition count:", df.rdd.getNumPartitions())

row_count = df.count()
print("Row count:", row_count)

summary_cols = ["Elevation", "Slope", "Horizontal_Distance_To_Roadways", "Cover_Type"]
df.select(*summary_cols).summary().show()


In [ ]:
assert set(FEATURE_COLUMNS + ["Cover_Type"]).issubset(df.columns)
assert df.filter(F.col("Cover_Type").isNull()).count() == 0

label_values = {int(r["Cover_Type"]) for r in df.select("Cover_Type").distinct().collect()}
assert label_values == {1, 2, 3, 4, 5, 6, 7}

class_balance_pdf = (
    df.groupBy("Cover_Type")
      .count()
      .orderBy("Cover_Type")
      .toPandas()
)
class_balance_pdf["rate"] = class_balance_pdf["count"] / class_balance_pdf["count"].sum()

duplicate_row_count = row_count - df.dropDuplicates().count()

print("Duplicate full rows:", duplicate_row_count)
display(class_balance_pdf)

In [ ]:

# Step 3: Visualize class balance
plt.figure(figsize=(8, 4))
plt.bar(class_balance_pdf["Cover_Type"].astype(str), class_balance_pdf["count"])
plt.title("Covertype class distribution")
plt.xlabel("Cover type")
plt.ylabel("Row count")
plt.show()


In [ ]:
modeling_df = (
    df.select(*FEATURE_COLUMNS, "Cover_Type")
      .repartition(TARGET_PARTITIONS)
      .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = modeling_df.count()

partition_sizes = modeling_df.rdd.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()
pd.Series(partition_sizes).describe()

In [ ]:

# Step 3: Plot partition-size distribution
plt.figure(figsize=(8, 4))
plt.hist(partition_sizes, bins=min(20, max(5, len(partition_sizes) // 2)))
plt.title("Partition-size distribution after repartition + persist")
plt.xlabel("Rows per partition")
plt.ylabel("Number of partitions")
plt.show()


### Post-execution interpretation questions

1. What does the class-distribution plot suggest about evaluation difficulty for minority versus majority classes?          
The plot should show that the Covertype classes are not equally represented. Minority classes will generally be harder to predict because the model has fewer examples to learn from, while majority classes can have a stronger influence on overall metrics.

2. What is a limitation of checking only full-row duplicates in this dataset? What additional checks might matter?      
Full-row duplicate checks only identify records where every column matches. They do not identify records that share the same features but have different labels, or repeated observations that may create other data-quality concerns. Checking duplicate feature combinations and inconsistent labels would provide additional information.

3. From a systems perspective, what does a very wide spread in `partition_sizes` imply about stragglers and throughput?     
A wide spread means that some partitions contain much more data than others. Larger partitions may take longer to process and become stragglers, reducing overall throughput because Spark must wait for slower tasks to finish.



> **Think like an ML Systems engineer:** 
> Validation is not finished when the schema prints correctly. You also need to ask whether the cluster can process the dataset evenly and repeatably.


### Step 4 — Lazy Evaluation, Actions, and Shuffle-Aware Execution

### Pre-coding questions

1. What happens when you define a chain of `map()` and `filter()` calls but never call an action?       
Spark builds a plan describing the transformations but does not actually execute them. An action such as count() is needed to trigger execution.

2. Why is `groupBy().agg(...)` usually more expensive than `filter()` or `select()` in a distributed system?        
A groupBy() often requires a shuffle because records with the same grouping key need to be brought together across partitions. A simple filter() or select() can usually operate within existing partitions without moving data between workers.

3. When would caching an intermediate result improve wall-clock time, and when might it be wasteful?        
Caching helps when the same intermediate result is used repeatedly because Spark does not need to recompute it each time. It can be wasteful when the result is only used once or is too large to fit comfortably in available memory.



### Function/class selection and parameter exploration

Use:

- `RDD.map`
- `RDD.filter`
- `RDD.count`
- `RDD.toDebugString`
- `DataFrame.explain`
- `DataFrame.groupBy`

Explore with `help()` or notebook `?`:

- `RDD.count()`
- `RDD.toDebugString()`
- `DataFrame.explain(mode="formatted")`

Explain the effect of:

- actions on lazy execution
- narrow versus wide transformations on shuffle cost
- `mode="formatted"` on plan readability
- repeated actions on cached versus uncached intermediates


In [ ]:

# Step 4: A simple timing helper
def timed_action(fn, label):
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f} seconds")
    return result, elapsed


In [ ]:
base_rdd = sc.parallelize(range(1, 100001), 8)
lazy_rdd = base_rdd.map(lambda x: x * x).filter(lambda x: x % 3 == 0)

debug_string = lazy_rdd.toDebugString()
print(debug_string.decode("utf-8") if isinstance(debug_string, bytes) else debug_string)
print("lazy_rdd count:", lazy_rdd.count())

narrow_df = modeling_df.select("Elevation", "Slope").filter(F.col("Elevation") > 2500)
wide_df = modeling_df.groupBy("Cover_Type").agg(F.avg("Elevation").alias("avg_elevation"))

print("Narrow plan")
narrow_df.explain(mode="formatted")

print("Wide plan")
wide_df.explain(mode="formatted")

In [ ]:

# Step 4: Compare repeated actions with and without caching on the same aggregation
agg_once, uncached_time = timed_action(
    lambda: modeling_df.groupBy("Cover_Type").agg(F.avg("Slope").alias("avg_slope")).count(),
    "Repeated aggregation (first action)"
)

cached_agg = modeling_df.groupBy("Cover_Type").agg(F.avg("Slope").alias("avg_slope")).cache()
_, cached_first = timed_action(lambda: cached_agg.count(), "Cached aggregation materialization")
_, cached_second = timed_action(lambda: cached_agg.count(), "Cached aggregation reuse")


### Post-execution interpretation questions

1. Which plan is more likely to contain an `Exchange` or shuffle boundary, and why?         
The wide aggregation plan using groupBy() is more likely to contain an Exchange because Spark needs to redistribute records by Cover_Type. The narrow plan using select() and filter() can usually process records within their existing partitions.

2. What is a limitation of using a tiny RDD example to reason about full ML pipelines?          
A small RDD example does not represent the memory usage, data volume, shuffles, and repeated computations involved in a full machine learning pipeline. It is useful for understanding lazy evaluation, but its performance cannot be directly compared with the full dataset.

3. From a systems perspective, what does the cached second action reveal about iterative workloads?         
The cached second action demonstrates how reusing stored results can reduce repeated computation. This is useful for iterative workloads where the same data or intermediate results are accessed multiple times.


### Step 5 — Leakage-Safe Baseline Pipeline

### Pre-coding questions

1. Why should feature assembly and scaling happen inside the fitted pipeline instead of in ad hoc preprocessing code run before the split?          
Keeping preprocessing inside the pipeline helps prevent data leakage because the transformations are fitted as part of the training process. The same fitted transformations can then be applied consistently to the test data and future data.

2. Why is multinomial logistic regression a reasonable baseline for this multiclass problem even if it may not be the strongest final model?        
It provides a relatively simple model for predicting the seven cover types. It is useful as a starting point because its performance and training cost can be compared with more complex models such as random forests.

3. What tradeoff appears when you increase `maxIter` without revisiting regularization or convergence diagnostics?      
Increasing maxIter gives the model more opportunities to converge, but it also increases training time. It does not automatically improve the model, especially if the regularization settings or other model choices are not appropriate.


In [ ]:

# Step 5: Train/test split
train_df, test_df = modeling_df.randomSplit([0.8, 0.2], seed=RANDOM_STATE)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


In [ ]:
label_indexer = StringIndexer(inputCol="Cover_Type", outputCol="label", handleInvalid="error")
assembler_lr = VectorAssembler(inputCols=FEATURE_COLUMNS, outputCol="raw_features")
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=False
)
lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    family="multinomial",
    maxIter=40,
    regParam=0.0
)

baseline_pipeline = Pipeline(stages=[label_indexer, assembler_lr, scaler, lr])

In [ ]:

# Step 5: Fit the baseline model
start = time.perf_counter()
baseline_model = baseline_pipeline.fit(train_df)
baseline_train_time = time.perf_counter() - start

baseline_predictions = baseline_model.transform(test_df).cache()
_ = baseline_predictions.count()

print(f"Baseline training time: {baseline_train_time:.3f} seconds")
baseline_predictions.select("Cover_Type", "label", "prediction", "probability").show(5, truncate=False)


In [ ]:
def metric_table(pred_df, model_name):
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
    metrics = {
        "accuracy": evaluator.evaluate(pred_df, {evaluator.metricName: "accuracy"}),
        "f1": evaluator.evaluate(pred_df, {evaluator.metricName: "f1"}),
        "weightedPrecision": evaluator.evaluate(pred_df, {evaluator.metricName: "weightedPrecision"}),
        "weightedRecall": evaluator.evaluate(pred_df, {evaluator.metricName: "weightedRecall"}),
    }
    return pd.DataFrame([{"model": model_name, **metrics}])

baseline_metrics_pdf = metric_table(baseline_predictions, "Multinomial Logistic Regression")
baseline_metrics_pdf

In [ ]:

# Step 5: Visualize baseline metrics
display(baseline_metrics_pdf)

metric_cols = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]
plt.figure(figsize=(8, 4))
plt.bar(metric_cols, baseline_metrics_pdf.loc[0, metric_cols])
plt.ylim(0, 1)
plt.title("Baseline multiclass metrics")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.show()


### Post-execution interpretation questions

1. Which metric is the best single summary here, and what important information still gets hidden?          
Weighted F1 is a useful single summary because it considers both precision and recall while accounting for class support. However, it can still hide poor performance on individual minority classes, so the confusion matrix and per-class metrics are also important.

2. What is a limitation of this linear baseline on Covertype, and what type of model might address it?          
Logistic regression has limitations when relationships between terrain features and cover types are nonlinear or involve complex feature interactions. A tree-based model such as a random forest can capture these patterns more effectively.

3. From a systems perspective, why might this baseline still be valuable even if its accuracy is not state of the art?      
The baseline is useful because it provides a simpler model that is easier to train, evaluate, and maintain. It also provides a reference point for deciding whether a more complex model provides enough improvement to justify its additional cost.


### Step 6 — Model Improvement with Random Forest + TrainValidationSplit

### Pre-coding questions

1. Why might a tree ensemble outperform the linear baseline on this dataset?        
A random forest can capture nonlinear relationships and interactions between the terrain features. It also combines multiple decision trees, which can improve predictive performance compared with a single linear model.

2. Why is `TrainValidationSplit` usually faster than `CrossValidator`, and what reliability tradeoff does that introduce?       
TrainValidationSplit evaluates each parameter combination using one training and validation split, while CrossValidator repeats the process across multiple folds. This makes TrainValidationSplit faster, but its results may depend more heavily on the particular validation split.

3. Which hyperparameters are most likely to increase both accuracy and systems cost in a random forest?         
Increasing numTrees and maxDepth can improve the model's ability to learn patterns, but both can increase training time, memory use, and model size. Larger trees and more trees can also increase prediction time.



### Function/class selection and parameter exploration

Use:

- `RandomForestClassifier`
- `ParamGridBuilder`
- `TrainValidationSplit`
- `MulticlassClassificationEvaluator`

Explore with `help()` or notebook `?`:

- `RandomForestClassifier(numTrees=..., maxDepth=..., featureSubsetStrategy=...)`
- `ParamGridBuilder().addGrid(...)`
- `TrainValidationSplit(trainRatio=..., parallelism=..., seed=...)`

Explain the effect of:

- `numTrees` on variance reduction versus runtime
- `maxDepth` on expressiveness versus overfitting and memory
- `featureSubsetStrategy` on tree diversity
- `parallelism` on tuning throughput and cluster utilization


In [ ]:
assembler_rf = VectorAssembler(inputCols=FEATURE_COLUMNS, outputCol="features")

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    seed=RANDOM_STATE,
    featureSubsetStrategy="sqrt"
)

rf_pipeline = Pipeline(stages=[label_indexer, assembler_rf, rf])

rf_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [10, 16])
    .build()
)

tvs = TrainValidationSplit(
    estimator=rf_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=rf_evaluator,
    trainRatio=0.8,
    parallelism=min(4, sc.defaultParallelism),
    seed=RANDOM_STATE
)

In [ ]:

# Step 6: Fit the tuned random-forest workflow
start = time.perf_counter()
tvs_model = tvs.fit(train_df)
rf_train_time = time.perf_counter() - start

best_pipeline_model = tvs_model.bestModel
rf_predictions = best_pipeline_model.transform(test_df).cache()
_ = rf_predictions.count()

rf_metrics_pdf = metric_table(rf_predictions, "Tuned Random Forest")
comparison_pdf = pd.concat([baseline_metrics_pdf, rf_metrics_pdf], ignore_index=True)

best_rf_stage = best_pipeline_model.stages[-1]
best_num_trees = best_rf_stage.getNumTrees() if callable(getattr(best_rf_stage, "getNumTrees", None)) else None
best_max_depth = best_rf_stage.getMaxDepth() if callable(getattr(best_rf_stage, "getMaxDepth", None)) else None

print(f"Random-forest training time: {rf_train_time:.3f} seconds")
print("Best numTrees:", best_num_trees)
print("Best maxDepth:", best_max_depth)
display(comparison_pdf)


In [ ]:

# Step 6: Compare metrics and training-time cost
display(comparison_pdf)

comparison_long = comparison_pdf.melt(
    id_vars="model",
    value_vars=["accuracy", "f1", "weightedPrecision", "weightedRecall"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(9, 4))
for model_name in comparison_long["model"].unique():
    subset = comparison_long[comparison_long["model"] == model_name]
    plt.plot(subset["metric"], subset["score"], marker="o", label=model_name)

plt.ylim(0, 1)
plt.title("Baseline versus tuned random forest")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend()
plt.show()

cost_pdf = pd.DataFrame([
    {"model": "Multinomial Logistic Regression", "train_seconds": baseline_train_time},
    {"model": "Tuned Random Forest", "train_seconds": rf_train_time},
])

plt.figure(figsize=(8, 4))
plt.bar(cost_pdf["model"], cost_pdf["train_seconds"])
plt.title("Training-time comparison")
plt.ylabel("Seconds")
plt.xticks(rotation=15)
plt.show()


### Post-execution interpretation questions

1. Which model would you choose if accuracy were the only objective, and what additional evidence would you still want before deployment?       
I would choose whichever model has the higher test accuracy, which would likely be the tuned random forest if it improves on the baseline. Before deployment, I would also want to review F1, weighted precision, weighted recall, the confusion matrix, training time, inference latency, and performance across individual classes.

2. What is a limitation of `TrainValidationSplit` for final model selection in a high-stakes setting?       
It relies on one validation split, so the selected model may perform differently with another split of the data. In a high-stakes setting, I would want additional validation, such as cross-validation and evaluation on an independent test dataset.

3. From a systems perspective, how should a team reason about a modest metric gain that requires a much larger training bill?       
The team should decide whether the improvement is large enough to justify the additional compute, memory, training time, and maintenance. If the performance gain is small, a simpler model may be a better choice because it is less expensive to operate.



> **Think like an ML Systems engineer:** Better accuracy is not automatically a better system. Improvement must be weighed against training cost, model size, latency, and operational complexity.


### Step 7 — Diagnostics: Confusion Patterns & Feature Importance

### Pre-coding questions

1. Why can weighted metrics look strong even when one or two classes are still problematic?         
Weighted metrics give more influence to classes with more examples. A model can perform well on the majority classes and still struggle with minority classes without the overall metrics showing how serious those problems are.

2. What kinds of relationships can feature importance suggest, and what can it not prove?       
Feature importance can suggest which features contribute most to the model's predictions. However, it cannot prove that a feature causes a particular cover type or explain the exact relationship between a feature and the target.

3. Which failure modes would you expect to remain even after switching from a linear model to a random forest?      
The random forest can still struggle with minority classes, overlapping feature patterns, noisy data, and changes in the data distribution. It can also overfit if its trees become too complex.


In [ ]:
label_model = best_pipeline_model.stages[0]
label_lookup = {float(i): int(lbl) for i, lbl in enumerate(label_model.labels)}

confusion_pdf = (
    rf_predictions.groupBy("label", "prediction")
    .count()
    .toPandas()
    .pivot(index="label", columns="prediction", values="count")
    .fillna(0)
    .sort_index()
    .sort_index(axis=1)
)
confusion_pdf = confusion_pdf.rename(index=label_lookup, columns=label_lookup)

error_pairs_pdf = (
    rf_predictions.filter(F.col("label") != F.col("prediction"))
    .groupBy("label", "prediction")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
    .toPandas()
)
error_pairs_pdf["label"] = error_pairs_pdf["label"].map(label_lookup)
error_pairs_pdf["prediction"] = error_pairs_pdf["prediction"].map(label_lookup)

display(confusion_pdf)
display(error_pairs_pdf)

In [ ]:

# Step 7: Plot the confusion matrix
plt.figure(figsize=(6, 5))
plt.imshow(confusion_pdf.values)
plt.title("Random-forest confusion matrix")
plt.xlabel("Predicted cover type")
plt.ylabel("True cover type")
plt.xticks(range(len(confusion_pdf.columns)), confusion_pdf.columns)
plt.yticks(range(len(confusion_pdf.index)), confusion_pdf.index)
plt.colorbar()
plt.show()

print("Top off-diagonal error pairs:")
display(error_pairs_pdf)


In [ ]:

# Step 7: Feature-importance diagnostics
rf_stage = best_pipeline_model.stages[-1]
importance_pdf = (
    pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "importance": rf_stage.featureImportances.toArray()
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_pdf.head(15))

plt.figure(figsize=(8, 5))
top_k = 15
subset = importance_pdf.head(top_k).sort_values("importance")
plt.barh(subset["feature"], subset["importance"])
plt.title("Top random-forest feature importances")
plt.xlabel("Importance")
plt.show()


### Post-execution interpretation questions

1. Which classes appear most confusable, and what domain explanation might account for that pattern?        
This should be answered using the confusion matrix and the top error pairs from the notebook. Classes with similar terrain characteristics may be confused because different forest cover types can occur in areas with similar elevation, slope, soil, and other environmental conditions.

2. What is a limitation of relying only on feature importance for next-step decisions? What would you add?          
Feature importance does not explain whether a feature has a positive or negative relationship with a prediction, and correlated features can affect the results. I would also use per-class performance, confusion patterns, and additional feature analysis to understand the model's behavior.

3. From a systems perspective, what is the cost of collecting many large diagnostics to pandas on the driver?       
Converting large Spark DataFrames to pandas transfers data to the driver and requires it to fit in driver memory. This can create a bottleneck, use significant memory, and potentially cause the driver to crash. Diagnostics should be limited or aggregated in Spark before being collected.


### Step 8 — Persistence, Monitoring, and Production Readiness

### Pre-coding questions

1. What can go wrong if the production scoring table arrives with a missing or reordered feature column?        
The pipeline may fail if a required feature is missing. If the features are supplied in the wrong order or mapped incorrectly, the model could make predictions using the wrong information and produce unreliable results.

2. Which signals would you log in production to detect data drift or performance regression?        
I would monitor the input schema, null rates by feature, prediction class distribution, latency, throughput, and delayed accuracy and F1 scores. Comparing feature distributions over time would also help identify changes in incoming data.

3. When is batch inference preferable to low-latency online inference for a Spark ML pipeline?      
Batch inference is preferable when predictions can be generated on a schedule and do not need to be returned immediately. It is useful for processing large datasets together because Spark can distribute the workload across the cluster.



### Function/class selection and parameter exploration

Use:

- `PipelineModel.write().save(...)`
- `PipelineModel.load(...)`
- `DataFrame.select`
- `DataFrame.limit`

Explore with `help()` or notebook `?`:

- `PipelineModel.write`
- `PipelineModel.load`
- `DataFrame.limit`

Explain the effect of:

- model save path choice on portability
- pipeline persistence on reproducibility
- batch-size choice on inference throughput
- monitoring granularity on observability cost


In [ ]:
best_pipeline_model.write().overwrite().save(MODEL_URI)
reloaded_model = PipelineModel.load(MODEL_URI)

batch_score_df = (
    reloaded_model
    .transform(test_df.limit(1000))
    .select("Cover_Type", "prediction", "probability")
)

batch_score_df.show(5, truncate=False)

In [ ]:

# Step 8: Build a lightweight monitoring checklist
monitoring_pdf = pd.DataFrame([
    {
        "signal": "Schema version",
        "why_log_it": "Detect feature additions, removals, or reordering before bad scoring happens",
        "granularity": "every batch / every deployment"
    },
    {
        "signal": "Null rate by feature",
        "why_log_it": "Detect upstream pipeline breakage or missing-source feeds",
        "granularity": "every batch"
    },
    {
        "signal": "Prediction class distribution",
        "why_log_it": "Detect concept drift or model collapse toward a dominant class",
        "granularity": "every batch"
    },
    {
        "signal": "Latency and throughput",
        "why_log_it": "Track service health and cluster-capacity regressions",
        "granularity": "every scoring job"
    },
    {
        "signal": "Delayed accuracy / F1",
        "why_log_it": "Detect true performance regression once labels arrive",
        "granularity": "daily / weekly"
    },
])

display(monitoring_pdf)


In [ ]:

# Step 8: A simple drift proxy using feature means on train vs. test for top RF features
top_features = importance_pdf.head(5)["feature"].tolist()

train_means = train_df.select([F.mean(c).alias(c) for c in top_features]).toPandas().iloc[0]
test_means = test_df.select([F.mean(c).alias(c) for c in top_features]).toPandas().iloc[0]

drift_proxy_pdf = pd.DataFrame({
    "feature": top_features,
    "train_mean": [float(train_means[c]) for c in top_features],
    "test_mean": [float(test_means[c]) for c in top_features],
})
drift_proxy_pdf["absolute_diff"] = (drift_proxy_pdf["train_mean"] - drift_proxy_pdf["test_mean"]).abs()

display(drift_proxy_pdf.sort_values("absolute_diff", ascending=False))


### Post-execution interpretation questions

1. Why is a saved full pipeline safer than saving only the fitted estimator stage?          
A full pipeline includes the fitted preprocessing stages and the estimator. Saving only the estimator could cause problems if future data is not transformed in exactly the same way as the training data.

2. What is a limitation of the simple mean-difference drift proxy used above? What stronger alternatives would you consider?        
Comparing feature means only shows differences in averages and can miss changes in distributions, outliers, and category frequencies. Stronger alternatives could include comparing feature distributions, checking categorical value frequencies, and using statistical drift tests.

3. From a systems perspective, which monitoring signals are most critical when labels arrive late or never arrive?      
Input schema, null rates, feature distributions, prediction class distribution, latency, and throughput become especially important. These signals can help identify data problems and model behavior changes even when actual labels are unavailable. Once labels arrive, accuracy and F1 should also be monitored.



## Reflection

Write a brief conclusion that answers the following:

1. Which model would you deploy for a nightly batch scoring job on Dataproc, and why?       
I would deploy the tuned random forest if it provides a meaningful improvement over logistic regression. A nightly batch job can handle the additional processing time more easily than a real-time application, making a stronger model worth considering. However, I would compare its performance and resource costs before making the final decision.

2. Where could leakage or reproducibility failure still occur in your notebook?         
Leakage could occur if preprocessing is performed using the full dataset before splitting or if test data influences model selection. Reproducibility problems could come from changes in dataset versions, Spark configurations, library versions, random seeds, or storage locations.

3. What is one change you would make if the dataset grew by 100× and had to remain in cloud storage for all processing?     
I would keep the dataset in cloud storage as Parquet and process it directly with Spark instead of loading the entire dataset into pandas on the driver. I would also review partition sizes, caching, and cluster resources to make sure the larger workload can be processed efficiently.


**End of Lab**